# Bronze Layer: Ingest Sensor Data from S3

This notebook uses **Auto Loader** (`cloudFiles` source) to continuously ingest sensor data files arriving on Amazon S3 into a Delta Lake table in the bronze layer.

In [0]:
# ── Configuration ──────────────────────────────────────────────
# Update these values to match your environment

# S3 source path where sensor data files are dropping
SOURCE_PATH = "s3://your-bucket/sensor-data/"

# File format of the source data (json, csv, parquet, text, ...)
FILE_FORMAT = "json"

# Target Delta table (catalog.schema.table)
TARGET_TABLE = "main.bronze.sensor_data"

# Checkpoint location for Auto Loader state (S3 or UC Volume path)
CHECKPOINT_PATH = "s3://your-bucket/checkpoints/sensor_data/"

# How often to process new files (default: process as soon as data arrives)
TRIGGER_PROCESSING_TIME = "10 seconds"

# ────────────────────────────────────────────────────────────────

In [0]:
# ── Configure Auto Loader (cloudFiles source) ─────────────────
# Auto Loader incrementally and efficiently processes new files
# as they arrive in cloud storage.

raw_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", FILE_FORMAT)
    .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "schema/")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.useNotifications", "false")  # directory listing mode
    .load(SOURCE_PATH)
)

# Add metadata columns for traceability
from pyspark.sql.functions import col, current_timestamp

bronze_df = (
    raw_df
    .withColumn("_source_file", col("_metadata.file_name"))
    .withColumn("_source_path", col("_metadata.file_path"))
    .withColumn("_ingest_ts", current_timestamp())
)

display(bronze_df)

In [0]:
# ── Create target table if it doesn't exist ────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE}
USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

# ── Start the streaming query ──────────────────────────────────
# foreachBatch gives us transactional, exactly-once writes

def upsert_to_delta(batch_df, batch_id):
    if batch_df.isEmpty():
        return
    (
        batch_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(TARGET_TABLE)
    )

query = (
    bronze_df.writeStream
    .foreachBatch(upsert_to_delta)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime=TRIGGER_PROCESSING_TIME)
    .queryName("sensor_data_autoloader")
    .start()
)

print(f"Streaming query '{query.name}' started — reading from {SOURCE_PATH} → {TARGET_TABLE}")
print(f"Check status: spark.streams.active for live queries")

In [0]:
# ── Monitor the active streaming query ─────────────────────────
import time

# Show recent progress for 3 iterations
for i in range(3):
    time.sleep(15)
    for q in spark.streams.active:
        latest = q.lastProgress
        if latest:
            print(f"[{q.name}] batch {latest.get('batchId')}: "
                  f"{latest.get('numInputRows', 0)} rows, "
                  f"{latest.get('processedRowsPerSecond', 0):.1f} rows/sec")
    print("---")

print("Query is running. Uncomment below to stop when done.")
# query.stop()